# Fine-tune and Evaluate OPT-125m on CNN/DailyMail

This notebook fine-tunes `facebook/opt-125m` without article chunking. Each article is processed once and the model generates exactly two top-level sections:

1. Summary
2. Aspects: Event, People, Date & Time, Location, Effects, and Countermeasures

The prompt is loaded from `data/processed/prompt_templates.json`, and the article is tokenized with the tokenizer loaded from the same pretrained OPT checkpoint. If a complete prompt would exceed OPT's context window, only the article token IDs are truncated; no chunks or intermediate summaries are created.

## Setup

In [1]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
PyTorch: 2.13.0+cu126
CUDA: 12.6
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [ ]:
# %pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import json
import time
from functools import partial

import pandas as pd
import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from rouge_score import rouge_scorer
import textstat

PROCESSED_DIR = "data/processed"
TRAIN_PATH = os.path.join(PROCESSED_DIR, "train.jsonl")
VAL_PATH = os.path.join(PROCESSED_DIR, "validation.jsonl")
TEST_PATH = os.path.join(PROCESSED_DIR, "test.jsonl")
TEMPLATES_PATH = os.path.join(PROCESSED_DIR, "prompt_templates.json")

MODEL_NAME = "facebook/opt-125m"
FINETUNED_MODEL_DIR = "models/opt-125m-cnn-news-simplifier"
RESULTS_DIR = "results"

MAX_SEQUENCE_LENGTH = 2048
MAX_NEW_TOKENS = 384
SEED = 42

os.makedirs(FINETUNED_MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator before training this notebook.")

DEVICE = "cuda"
COMPUTE_DTYPE = (
    torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)
print(f"Using device: {DEVICE}; compute dtype: {COMPUTE_DTYPE}")

W0802 15:43:25.169000 120656 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Using device: cuda; compute dtype: torch.bfloat16


## Load train, validation and test splits

In [ ]:
def load_jsonl(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Could not find '{path}'. Run preprocessing.ipynb and "
            "split_data.ipynb first."
        )

    records = []
    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    frame = pd.DataFrame(records)
    required_columns = {
        "article_id",
        "article",
        "summary",
        "article_word_count",
        "aspects",
    }
    missing_columns = required_columns - set(frame.columns)
    if missing_columns:
        raise ValueError(
            f"{path} is missing columns: {sorted(missing_columns)}"
        )
    if frame["article_id"].duplicated().any():
        raise ValueError(f"{path} contains duplicate article IDs.")

    return frame


train_df = load_jsonl(TRAIN_PATH)
val_df = load_jsonl(VAL_PATH)
test_df = load_jsonl(TEST_PATH)

print(f"Train      : {len(train_df)}")
print(f"Validation : {len(val_df)}")
print(f"Test       : {len(test_df)}")

Train:      240
Validation: 30
Test:       30


## Build direct prompts and tokenize with the OPT tokenizer

The prompt template is loaded from `prompt_templates.json`. The prompt, article, and target are encoded by the tokenizer from the exact `facebook/opt-125m` checkpoint. Prompt tokens are masked with `-100`, so the causal-language-model loss is calculated only on the desired response.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

if not os.path.exists(TEMPLATES_PATH):
    raise FileNotFoundError(f"Could not find '{TEMPLATES_PATH}'.")

with open(TEMPLATES_PATH, "r", encoding="utf-8") as file:
    prompt_templates = json.load(file)

if "prompt_template" not in prompt_templates:
    raise KeyError("prompt_templates.json must contain 'prompt_template'.")

PROMPT_TEMPLATE = prompt_templates["prompt_template"]
if PROMPT_TEMPLATE.count("{article}") != 1:
    raise ValueError(
        "The prompt template must contain exactly one '{article}' placeholder."
    )

PROMPT_PREFIX, PROMPT_SUFFIX = PROMPT_TEMPLATE.split("{article}")

ASPECT_LABELS = {
    "event": "Event",
    "people": "People",
    "date_time": "Date & Time",
    "location": "Location",
    "effects": "Effects",
    "countermeasures": "Countermeasures",
}


def format_aspect_value(value) -> str:
    if isinstance(value, list):
        values = [str(item).strip() for item in value if str(item).strip()]
        return "; ".join(values) if values else "Not stated."
    if value is None or not str(value).strip():
        return "Not stated."
    return str(value).strip()


def build_target(summary: str, aspects: dict) -> str:
    if not isinstance(aspects, dict):
        raise TypeError("Each 'aspects' value must be a dictionary.")

    lines = [
        "Summary:",
        str(summary).strip(),
        "Aspects:",
    ]
    lines.extend(
        f"{label}: {format_aspect_value(aspects.get(key))}"
        for key, label in ASPECT_LABELS.items()
    )
    return "\n".join(lines)


prefix_ids = tokenizer(
    PROMPT_PREFIX,
    add_special_tokens=True,
)["input_ids"]
suffix_ids = tokenizer(
    PROMPT_SUFFIX,
    add_special_tokens=False,
)["input_ids"]


def encode_prompt(article: str, max_prompt_tokens: int):
    article_ids = tokenizer(
        str(article),
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    available_article_tokens = (
        max_prompt_tokens - len(prefix_ids) - len(suffix_ids)
    )
    if available_article_tokens < 1:
        raise ValueError("The response target leaves no room for the article.")

    used_article_ids = article_ids[:available_article_tokens]
    prompt_ids = prefix_ids + used_article_ids + suffix_ids

    return prompt_ids, {
        "article_token_count": len(article_ids),
        "used_article_token_count": len(used_article_ids),
        "article_was_truncated": len(used_article_ids) < len(article_ids),
    }


def encode_training_record(record: dict):
    target_text = build_target(record["summary"], record["aspects"])
    target_ids = tokenizer(
        target_text,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"] + [tokenizer.eos_token_id]

    prompt_budget = MAX_SEQUENCE_LENGTH - len(target_ids)
    prompt_ids, token_info = encode_prompt(
        record["article"],
        max_prompt_tokens=prompt_budget,
    )

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }, token_info


class NewsSimplificationDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.examples = []
        self.token_details = []

        for record in frame.to_dict(orient="records"):
            example, token_info = encode_training_record(record)
            self.examples.append(example)
            self.token_details.append(token_info)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


def collate_fn(batch, pad_token_id: int):
    max_len = max(len(item["input_ids"]) for item in batch)

    input_ids = []
    attention_masks = []
    labels = []

    for item in batch:
        pad_length = max_len - len(item["input_ids"])
        input_ids.append(item["input_ids"] + [pad_token_id] * pad_length)
        attention_masks.append(item["attention_mask"] + [0] * pad_length)
        labels.append(item["labels"] + [-100] * pad_length)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


train_dataset = NewsSimplificationDataset(train_df)
val_dataset = NewsSimplificationDataset(val_df)

train_truncated = sum(
    item["article_was_truncated"]
    for item in train_dataset.token_details
)
val_truncated = sum(
    item["article_was_truncated"]
    for item in val_dataset.token_details
)

print(f"Tokenizer checkpoint: {tokenizer.name_or_path}")
print(f"Training examples: {len(train_dataset)}; truncated: {train_truncated}")
print(f"Validation examples: {len(val_dataset)}; truncated: {val_truncated}")

Tokenizer checkpoint: facebook/opt-125m
Training examples: 240; truncated: 0
Validation examples: 30; truncated: 0


## Load OPT-125m and configure QLoRA

In [6]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
)

print("Model context limit:", model.config.max_position_embeddings)
if MAX_SEQUENCE_LENGTH > model.config.max_position_embeddings:
    raise ValueError("MAX_SEQUENCE_LENGTH exceeds the model context limit.")

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.config.use_cache = False
model.print_trainable_parameters()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Model context limit: 2048
trainable params: 589,824 || all params: 125,829,120 || trainable%: 0.4688


In [7]:
training_args = TrainingArguments(
    output_dir=os.path.join(FINETUNED_MODEL_DIR, "checkpoints"),
    seed=SEED,
    num_train_epochs=10,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=COMPUTE_DTYPE == torch.float16,
    bf16=COMPUTE_DTYPE == torch.bfloat16,
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    report_to=[],
)

safe_collator = partial(
    collate_fn,
    pad_token_id=tokenizer.pad_token_id,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=safe_collator,
)

train_result = trainer.train()
print(train_result)

Epoch,Training Loss,Validation Loss
1,2.261014,2.258980
2,2.129352,2.120903
3,1.955497,2.039685
4,1.917222,1.987045
5,1.950496,1.953196
6,1.871175,1.929079
7,1.853476,1.913724
8,1.859284,1.901483
9,1.838346,1.895340
10,1.823591,1.893172


d:\User\Documents\GitHub\news_simplification_assistant\.venv\Lib\site-packages\peft\utils\other.py:1496: UserWarning: Unable to fetch remote file due to the following error [WinError 10054] An existing connection was forcibly closed by the remote host - silently ignoring the lookup for the file config.json in facebook/opt-125m.
  warnings.warn(
d:\User\Documents\GitHub\news_simplification_assistant\.venv\Lib\site-packages\peft\utils\save_and_load.py:438: UserWarning: Could not find a config file in facebook/opt-125m - will assume that the vocabulary was not modified.
  warnings.warn(
d:\User\Documents\GitHub\news_simplification_assistant\.venv\Lib\site-packages\peft\utils\other.py:1496: UserWarning: Unable to fetch remote file due to the following error [WinError 10054] An existing connection was forcibly closed by the remote host - silently ignoring the lookup for the file config.json in facebook/opt-125m.
  warnings.warn(
d:\User\Documents\GitHub\news_simplification_assistant\.venv\L

TrainOutput(global_step=300, training_loss=1.9824294408162435, metrics={'train_runtime': 345.2111, 'train_samples_per_second': 6.952, 'train_steps_per_second': 0.869, 'total_flos': 930072538644480.0, 'train_loss': 1.9824294408162435, 'epoch': 10.0})


## Save the fine-tuned adapter and exact tokenizer

In [8]:
trainer.save_model(FINETUNED_MODEL_DIR)
tokenizer.save_pretrained(FINETUNED_MODEL_DIR)
print(f"Saved model adapter and tokenizer to '{FINETUNED_MODEL_DIR}'")

Saved model adapter and tokenizer to 'models/opt-125m-cnn-news-simplifier'
